In [1]:
# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
import os

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# mostra o dataframe para caber certinho na tela
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [2]:
# acessar a pasta experiments
dir_experiments="../experiments/prototypeEvaluation_filterReduction/results"

# Estrutura para armazenar os caminhos dos arquivos 
results_files = defaultdict(dict)

for folder in sorted(os.listdir(dir_experiments)):
    path = os.path.join(dir_experiments, folder)
    for file in sorted(os.listdir(path)):
       if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[folder][file] = os.path.join(path, file)
           

In [3]:
results_data = []

for folder, contents in results_files.items():
    superpixel = int(folder.split("_")[0].replace('super', ''))
    nimage = int(folder.split("_")[1].replace('images', ''))
    technique = folder.split('_')[2]
    # print(f"{folder=}")
    # print(f"{superpixel=}, {nimage=}, {technique=}")
    for file, dir in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = dir
            seed_value = int(file.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class_accuracies = list(map(float, lines[0].strip().split(';')[:9]))
                    class1_accuracy, class2_accuracy = class_accuracies[0], class_accuracies[1]
                    class3_accuracy, class4_accuracy = class_accuracies[2], class_accuracies[3]
                    class5_accuracy, class6_accuracy = class_accuracies[4], class_accuracies[5]
                    class7_accuracy, class8_accuracy = class_accuracies[6], class_accuracies[7]
                    class9_accuracy = class_accuracies[8]
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'technique': technique,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'class3_accuracy': class3_accuracy,
                        'class4_accuracy': class4_accuracy,
                        'class5_accuracy': class5_accuracy,
                        'class6_accuracy': class6_accuracy,
                        'class7_accuracy': class7_accuracy,
                        'class8_accuracy': class8_accuracy,
                        'class9_accuracy': class9_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })


# Convert the results list to a DataFrame
df_technique = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file for further analysis
# df_technique.to_csv('prototype_results_summary.csv', index=False)

In [4]:
df_technique.head(100)

,superpixel,nimage,technique,seed,class1_accuracy,class2_accuracy,class3_accuracy,class4_accuracy,class5_accuracy,class6_accuracy,class7_accuracy,class8_accuracy,class9_accuracy,kappa,global_accuracy,nfeat
0,150,2,cossine,1011,0.954023,0.975,0.783784,0.737705,0.680473,0.941489,0.868852,0.932203,0.749402,0.662456,0.788424,25392
1,150,2,cossine,1213,0.954023,0.975,0.783784,0.737705,0.680473,0.941489,0.868852,0.932203,0.749402,0.662456,0.788424,25392
2,150,2,cossine,123,0.954023,0.925,0.756757,0.901639,0.846154,0.962766,0.819672,0.915254,0.822368,0.750713,0.849042,25392
3,150,2,cossine,2735,0.959770,0.925,0.702703,0.852459,0.668639,0.893617,0.770492,0.771186,0.584928,0.514682,0.666797,25392
4,150,2,cossine,42,0.890805,0.900,0.716216,0.836066,0.739645,0.930851,0.885246,0.881356,0.873206,0.765910,0.865467,25392
5,150,2,cossine,456,0.804598,0.850,0.554054,0.852459,0.745562,0.925532,0.754098,0.889831,0.722488,0.606537,0.753226,25392
6,150,2,cossine,6854,0.810345,0.925,0.662162,0.803279,0.857988,0.920213,0.803279,0.881356,0.831340,0.722159,0.835745,25392
7,150,2,cossine,7580,0.758621,0.775,0.675676,0.508197,0.497041,0.776596,0.704918,0.762712,0.653708,0.473875,0.664842,25392
8,150,2,cossine,789,0.798851,0.900,0.716216,0.688525,0.597633,0.877660,0.721311,0.728814,0.541866,0.444570,0.614783,25392
9,150,2,cossine,8900,0.827586,0.825,0.702703,0.836066,0.763314,0.941489,0.770492,0.898305,0.628588,0.552023,0.700039,25392


In [5]:
# Define the metrics to be analyzed (excluding 'nfeat' for summary statistics)
metrics = ['class1_accuracy', 'class2_accuracy', 'class3_accuracy', 'class4_accuracy', 'class5_accuracy', 'class6_accuracy', 'class7_accuracy', 'class8_accuracy', 'class9_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Agrupa por superpixel, nimage e technique e calcula média e desvio padrão para cada métrica
summary_stats = df_technique.groupby(['superpixel', 'nimage', 'technique'])[metrics[:-1]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# Arrange the DataFrame according to the order of superpixel values
# superpixels_values = sorted(superpixels_values)
# summary_stats = summary_stats.set_index('superpixel_').loc[superpixels_values].reset_index()

# Select only the metrics of interest for visualization and highlight the highest values
summary_stats = summary_stats[['superpixel_', 'nimage_', 'technique_', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
    subset=['kappa_mean', 'global_accuracy_mean'], color='gray'
)

# Format values as percentages for better presentation
summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Display the styled DataFrame
summary_stats

,superpixel_,nimage_,technique_,kappa_mean,kappa_std,global_accuracy_mean,global_accuracy_std
0,150,2,cossine,61.5538%,0.115586,75.2679%,0.087140
1,150,2,euclidean,71.9724%,0.069896,83.1404%,0.047821
2,150,3,cossine,69.1112%,0.083671,81.0168%,0.061827
3,150,3,euclidean,69.5311%,0.093215,81.2436%,0.069599
4,150,4,cossine,70.8736%,0.079479,82.3582%,0.052849
5,150,4,euclidean,70.6686%,0.065953,82.2409%,0.047225
